# Preprocessing airline passenger data

Split before fitting transformations so validation data does not influence imputation, scaling, or encoding. Keep the fitted pipeline for future predictions.

## 1. Load data


In [6]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

print("Training data:", df.shape)
print("Test data:", test_df.shape)
display(df.head())

Training data: (699635, 23)
Test data: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,1,3,4,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,5,5,5,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,3,4,3,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,3,4,5,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,5,5,4,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


## 2. Prepare unrated service responses

A service rating of 0 means not rated. Create a binary `_not_rated` indicator (1 = not rated, 0 = rated), then replace zeros with missing values in copies of the training and test data. Only service-rating columns are changed; zero delays remain valid.

After the train/validation split, fill missing ratings with each column's median learned from rated training responses only. Apply the same medians to validation and test. This step prepares the indicators but deliberately leaves median imputation pending until that split. Indicators are added for every rating to keep the schema consistent.

In [8]:
rating_columns = [
    "Inflight wifi service", "Departure/Arrival time convenient",
    "Ease of Online booking", "Gate location", "Food and drink",
    "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling",
    "Checkin service", "Cleanliness",
]

# Work on copies so the original data remains available.
df_prepared = df.copy()
test_prepared = test_df.copy()

for data in [df_prepared, test_prepared]:
    for col in rating_columns:
        data[f"{col}_not_rated"] = data[col].eq(0).astype("int8")
        data[col] = data[col].mask(data[col].eq(0))

rating_summary = pd.DataFrame({
    "train_not_rated": df[rating_columns].eq(0).sum(),
    "train_not_rated_percent": df[rating_columns].eq(0).mean().mul(100).round(2),
    "test_not_rated": test_df[rating_columns].eq(0).sum(),
})
display(rating_summary)
data.head()


,train_not_rated,train_not_rated_percent,test_not_rated
Inflight wifi service,13102,1.87,5682
Departure/Arrival time convenient,25751,3.68,10889
Ease of Online booking,16073,2.30,6797
Gate location,1455,0.21,617
Food and drink,1014,0.14,493
Online boarding,9147,1.31,3809
Seat comfort,838,0.12,399
Inflight entertainment,1020,0.15,417
On-board service,801,0.11,359
Leg room service,1693,0.24,730


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,Inflight wifi service_not_rated,Departure/Arrival time convenient_not_rated,Ease of Online booking_not_rated,Gate location_not_rated,Food and drink_not_rated,Online boarding_not_rated,Seat comfort_not_rated,Inflight entertainment_not_rated,On-board service_not_rated,Leg room service_not_rated,Baggage handling_not_rated,Checkin service_not_rated,Cleanliness_not_rated
0,699635,25,1197,4.0,4.0,4.0,2.0,3.0,4.0,3.0,3.0,2.0,5.0,2,1.0,3.0,0,0.0,Female,disloyal Customer,Business travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
1,699636,36,829,5.0,5.0,5.0,5.0,5.0,4.0,5.0,5.0,1.0,2.0,4,3.0,5.0,0,0.0,Male,Loyal Customer,Business travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
2,699637,45,472,4.0,5.0,5.0,5.0,4.0,4.0,4.0,4.0,3.0,3.0,1,3.0,4.0,0,0.0,Male,Loyal Customer,Business travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
3,699638,45,762,2.0,2.0,2.0,5.0,3.0,2.0,3.0,3.0,3.0,4.0,4,4.0,3.0,0,0.0,Female,disloyal Customer,Business travel,Business,0,0,0,0,0,0,0,0,0,0,0,0,0
4,699639,34,594,1.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,2.0,4.0,2,1.0,2.0,0,0.0,Male,Loyal Customer,Personal Travel,Eco Plus,0,0,0,0,0,0,0,0,0,0,0,0,0


## 3. Fill missing arrival delays

Calculate the arrival-delay median from `df_prepared`, then use it to fill missing arrival delays in `df_prepared` and `test_prepared`. Existing values, including zero delays, remain unchanged. No split is performed. For later validation, recalculate the median using only the training subset before filling its validation subset.

In [10]:
arrival_column = "Arrival Delay in Minutes"
arrival_median = df_prepared[arrival_column].median()
if pd.isna(arrival_median):
    raise ValueError("Cannot calculate the median: all arrival delays are missing.")

fill_summary = []
for name, data in [("Training data", df_prepared), ("Test data", test_prepared)]:
    missing_before = data[arrival_column].isna().sum()
    data[arrival_column] = data[arrival_column].fillna(arrival_median)
    fill_summary.append({"dataset": name, "filled_count": missing_before,
                         "missing_after": data[arrival_column].isna().sum()})

print(f"Arrival-delay median: {arrival_median:g} minutes")
display(pd.DataFrame(fill_summary).set_index("dataset"))


Arrival-delay median: 0 minutes


,filled_count,missing_after
dataset,,
Training data,292,0
Test data,130,0


## 4. Stratified train/validation split

Reserve 20% for validation, preserving satisfaction proportions with `stratify=y`. Use `random_state=42` for reproducibility. Exclude `id` and separate the 0/1 target. The external test data stays separate.

Recalculate arrival-delay imputation from original values using training indices only, replacing the earlier full-data imputation in the modeling subsets. Fit subsequent rating imputation, scaling, and encoding only on `X_train`.

In [12]:
from sklearn.model_selection import train_test_split

X = df_prepared.drop(columns=["id", "satisfaction"])
y = df_prepared["satisfaction"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train = X_train.copy()
X_val = X_val.copy()
X_test = test_prepared[X.columns].copy()
test_ids = test_prepared["id"].copy()

# Recalculate arrival-delay imputation from the training subset only.
# Use original values because the previous cell already filled the prepared data.
arrival_column = "Arrival Delay in Minutes"
arrival_median = df.loc[X_train.index, arrival_column].median()
for data in [X_train, X_val]:
    data[arrival_column] = df.loc[data.index, arrival_column].fillna(arrival_median)
X_test[arrival_column] = test_df[arrival_column].fillna(arrival_median)

split_summary = pd.DataFrame({
    "all_count": y.value_counts(),
    "train_count": y_train.value_counts(),
    "validation_count": y_val.value_counts(),
    "all_percent": y.value_counts(normalize=True).mul(100),
    "train_percent": y_train.value_counts(normalize=True).mul(100),
    "validation_percent": y_val.value_counts(normalize=True).mul(100),
}).sort_index()
split_summary.index.name = "satisfaction"
display(split_summary.round(2))
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Training-only arrival-delay median:", arrival_median)


,all_count,train_count,validation_count,all_percent,train_percent,validation_percent
satisfaction,,,,,,
0,389296,311437,77859,55.64,55.64,55.64
1,310339,248271,62068,44.36,44.36,44.36


Training: (559708, 34)
Validation: (139927, 34)
Training-only arrival-delay median: 0.0


In [13]:
X.head()

,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,Inflight wifi service_not_rated,Departure/Arrival time convenient_not_rated,Ease of Online booking_not_rated,Gate location_not_rated,Food and drink_not_rated,Online boarding_not_rated,Seat comfort_not_rated,Inflight entertainment_not_rated,On-board service_not_rated,Leg room service_not_rated,Baggage handling_not_rated,Checkin service_not_rated,Cleanliness_not_rated
0,36,694,4.0,4.0,4.0,3.0,1.0,4.0,1.0,1.0,3.0,4.0,4,3.0,1.0,0,0.0,Female,Loyal Customer,Personal Travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
1,56,651,5.0,5.0,5.0,5.0,5.0,4.0,4.0,5.0,5.0,5.0,5,4.0,3.0,0,0.0,Male,Loyal Customer,Business travel,Business,0,0,0,0,0,0,0,0,0,0,0,0,0
2,31,1371,3.0,4.0,2.0,5.0,3.0,3.0,3.0,3.0,4.0,3.0,4,5.0,3.0,0,0.0,Female,Loyal Customer,Personal Travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
3,10,1616,3.0,5.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,5.0,3,4.0,3.0,0,0.0,Male,Loyal Customer,Personal Travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
4,23,481,3.0,4.0,3.0,4.0,5.0,3.0,5.0,5.0,5.0,4.0,5,3.0,5.0,5,0.0,Female,disloyal Customer,Business travel,Eco,0,0,0,0,0,0,0,0,0,0,0,0,0
